# Agent Fundamentals: The Perception-Action Loop

An AI agent perceives its environment, reasons about what to do, and takes actions that change the environment. This loop — perceive, reason, act — is the foundation of all agent architectures, from 1960s rule-based dialogue systems to modern LLM agents.

## A Brief History of Dialogue Agents

The agent idea has a long history in AI:

**ELIZA** (Weizenbaum, 1966): pattern-matching chatbot that reflected user statements back as questions. Users formed emotional attachments despite knowing it was a program — the 'ELIZA effect' warned researchers about anthropomorphization.

**ALICE** (Wallace, 1995): AIML-based chatbot with thousands of hand-crafted rules. Won the Loebner Prize three times. Brittle — any input outside its rule set produced generic fallbacks.

**Rule-based dialogue systems** (2000s): task-oriented dialogue systems with hand-crafted state machines. Worked reliably within narrow domains (airline booking, customer support) but required months of engineering per domain and broke on out-of-domain inputs.

The shift to neural systems eliminated the hand-crafting bottleneck but introduced new challenges: stochastic behavior, hallucination, and difficulty specifying constraints.

## Modern Agent Taxonomy

Modern LLM agents are classified by their architecture:

**Single-agent**: one model in a perceive-reason-act loop. Simplest architecture; appropriate for well-scoped tasks.

**Multi-agent**: multiple models coordinating. Enables parallelism, specialization, and mutual verification.

**Tool-augmented**: agent has access to external tools (search, code execution, databases). Extends capability beyond parametric knowledge.

**Memory-augmented**: agent maintains state across turns via in-context memory, vector stores, or external databases.

The key design decisions: synchronous vs asynchronous action, centralized vs distributed planning, push vs pull memory access.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import json

@dataclass
class Observation:
    content: str
    source: str
    timestamp: int = 0

@dataclass
class Action:
    action_type: str  # 'tool_call', 'message', 'terminate'
    content: str
    tool_name: Optional[str] = None
    tool_args: Optional[dict] = None

@dataclass
class AgentStep:
    observation: Observation
    reasoning: str
    action: Action
    step_num: int

class MinimalAgent:
    def __init__(self, model_fn: Callable, tools: dict = None, max_steps: int = 10):
        self.model = model_fn
        self.tools = tools or {}
        self.max_steps = max_steps
        self.history: list = []

    def perceive(self, observation: Observation):
        self.history.append({'role': 'observation', 'content': observation.content,
                              'source': observation.source})

    def reason_and_act(self) -> Action:
        context = self._build_context()
        response = self.model(context)
        return self._parse_action(response)

    def _build_context(self) -> str:
        parts = ['You are a helpful agent. Use available tools to complete tasks.']
        if self.tools:
            tool_desc = '\n'.join(f'- {k}: {v["description"]}' for k, v in self.tools.items())
            parts.append(f'Available tools:\n{tool_desc}')
        for h in self.history[-10:]:
            parts.append(f"[{h['role']}] {h['content'][:100]}")
        return '\n'.join(parts)

    def _parse_action(self, response: str) -> Action:
        import re
        tool_match = re.search(r'TOOL:(\w+)\((.*)\)', response, re.DOTALL)
        if tool_match:
            name, args_str = tool_match.group(1), tool_match.group(2)
            try:
                args = json.loads(args_str) if args_str.strip() else {}
            except Exception:
                args = {'query': args_str}
            return Action('tool_call', response[:80], tool_name=name, tool_args=args)
        if 'DONE:' in response:
            answer = response.split('DONE:')[-1].strip()
            return Action('terminate', answer)
        return Action('message', response[:200])

    def execute_action(self, action: Action) -> Optional[str]:
        if action.action_type == 'tool_call' and action.tool_name in self.tools:
            try:
                result = self.tools[action.tool_name]['fn'](**action.tool_args)
                return str(result)
            except Exception as e:
                return f'Tool error: {e}'
        return None

    def run(self, task: str) -> str:
        self.perceive(Observation(task, 'user', 0))
        for step in range(self.max_steps):
            action = self.reason_and_act()
            self.history.append({'role': 'agent', 'content': action.content})
            if action.action_type == 'terminate':
                return action.content
            if action.action_type == 'tool_call':
                result = self.execute_action(action)
                if result:
                    self.perceive(Observation(result, action.tool_name, step))
        return 'Max steps reached'

# Demo
step_n = [0]
def simple_model(context):
    step_n[0] += 1
    if step_n[0] == 1:
        return 'TOOL:calculator({"a": 15, "b": 0.20})'
    return 'DONE: 15% tip on $80 is $12.00'

agent = MinimalAgent(
    model_fn=simple_model,
    tools={'calculator': {'fn': lambda a, b: a * b, 'description': 'Multiply two numbers'}},
    max_steps=5
)
result = agent.run('What is a 15% tip on an $80 meal?')
print(f'Task result: {result}')
print(f'Steps taken: {len([h for h in agent.history if h["role"]=="agent"])}')

## The Agent Loop in Practice

The perception-action loop in production agents has several practical concerns:

**Termination conditions**: agents need clear stop conditions — reaching the goal, exhausting budget, or detecting an unrecoverable error. Without explicit termination, agents can loop indefinitely.

**Error handling**: tool failures, unexpected observations, and model refusals all interrupt the loop. Robust agents retry with backoff, escalate to a fallback strategy, or surface the error to the user.

**Observability**: every step must be logged with full context for debugging. Silent failures in long agent runs are very difficult to diagnose post-hoc.

**Idempotency**: for agents that take external actions (send email, write file, call API), rerunning a step must not cause double-effects. Production agents checkpoint state and skip already-executed actions on retry.